# 5b – Initial-Condition Analysis
## BruteForce vs JRA55-FOSIRL coupled restart files

This notebook orchestrates the `workflows/diagnostics/initial_conditions/` workflow and provides
interactive exploration of results.  It is the companion to
**5a\_refactor\_drift\_analysis.ipynb**, which analyses monthly forecast adjustment.

### Recommended progression

1. Run the pilot date (`1980-05-01-00000`) end-to-end.
2. Confirm atmosphere checksums are IDENTICAL (control check).
3. Identify which non-atmospheric physical variables actually differ.
4. Develop detailed diagnostics for those variables.
5. Scale to May and November starts.
6. Relate IC differences to early adjustment from `5a`.

> ℹ️ A checksum difference alone may result from metadata.
> Step 2 separates physical-state variables from bookkeeping variables.
> Step 3 confirms whether the *values* differ.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
import esp_lab

# Development-checkout fallback: a clean install exposes ``workflows`` directly.
REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.initial_conditions import check_physical_consistency as physical_consistency_workflow
from workflows.diagnostics.initial_conditions import compare_netcdf_structure as structure_comparison_workflow
from workflows.diagnostics.initial_conditions import compute_ic_statistics as statistics_workflow
from workflows.diagnostics.initial_conditions import config as ic_config
from workflows.diagnostics.initial_conditions import inventory_and_hash as inventory_workflow
from workflows.diagnostics.initial_conditions import plot_component_differences as plotting_workflow
from workflows.diagnostics.initial_conditions import summarize_campaign as campaign_summary_workflow

from esp_lab.diagnostics import (
    ICConfig, ExperimentPair, ComponentSpec, DEFAULT_COMPONENTS,
    AuditResult, ic_variable_stats, aggregate_campaign_stats,
    check_atm_surface_vs_land, check_ocean_ice_consistency,
    write_manifest_json,
    discover_start_dates, match_start_dates,
    build_file_manifest, write_audit_csv, load_audit_csv,
    open_restart_file,
)
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster

load_config = ic_config.load_config
build_ic_config = ic_config.build_ic_config
IC_DIR = Path(ic_config.__file__).resolve().parent
print('ESP-Lab IC analysis imports OK')


## Dask Setup

In [ ]:
machine_env = os.environ.get('CLUSTER_TYPE', 'local')

dask_cfg = DaskConfig(
    cluster_type=machine_env,
    workers=8,
    cores=4,
    memory='16GB',
    walltime='04:00:00',
)

cluster, client = get_cluster_client(dask_cfg)
print(client)


## User Configuration

**All** parameters are defined here.  Edit this cell; no other cell below
needs to change.


In [ ]:
# ── Load from config.yaml ───────────────────────────────────────
CONFIG_PATH = IC_DIR / 'config.yaml'
cfg_dict    = load_config(CONFIG_PATH)

# ── Pilot mode toggle ────────────────────────────────────────────
# Set PILOT_ONLY = False to process all start dates.
PILOT_ONLY = True
PILOT_DATE = '1980-05-01-00000'
FORCE_RECOMPUTE = False  # True only when source IC files have changed

# ── Build ICConfig ───────────────────────────────────────────────
ic_cfg = build_ic_config(cfg_dict, pilot_only=PILOT_ONLY)
if PILOT_ONLY:
    ic_cfg.pilot_date = PILOT_DATE

print('Active dates :', ic_cfg.active_dates)
print('Components   :', [c.name for c in ic_cfg.components])
print('ref root     :', ic_cfg.experiment_pair.ref_root)
print('test root    :', ic_cfg.experiment_pair.test_root)

# ── Output directories ───────────────────────────────────────────
OUT_ROOT     = Path(ic_cfg.output_root)
FIGURE_OUTDIR = Path(cfg_dict['output']['figure_outdir'])
MANIFEST_DIR = OUT_ROOT / 'manifests'
FC_DIR       = OUT_ROOT / 'file_comparison'
VS_DIR       = OUT_ROOT / 'variable_statistics'
MAPS_DIR     = FIGURE_OUTDIR / 'maps'
PROFILES_DIR = FIGURE_OUTDIR / 'profiles'
CS_DIR       = OUT_ROOT / 'campaign_summary'


## Step 1 — Inventory and Hash

Discover matched start-date directories and compute SHA-256 checksums.

* Atmospheric EN00–EN09: all members hashed (control check).
* Non-atmospheric: one hash per start date.

**Expected result** for pilot: atmosphere files IDENTICAL, non-atmospheric TBD.


In [ ]:
%%time

audit_df = inventory_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    compute_hash=True,
    reuse_existing=not FORCE_RECOMPUTE,
    verbose=True,
)
IC_DATA_READY = not audit_df.empty
if IC_DATA_READY:
    display(audit_df.groupby(['component', 'status']).size().unstack(fill_value=0))
else:
    print('[STOP] No matched IC files were found. Verify the configured roots and active date.')


### Atmospheric control check

In [ ]:
if IC_DATA_READY:
    manifest_csv = MANIFEST_DIR / f"{ic_cfg.active_dates[0]}.csv"
    manifest = load_audit_csv(manifest_csv)
    atm_rows = manifest[manifest['component'] == 'atm']
    print('Atmospheric checksums:')
    print(atm_rows[['member', 'status', 'ref_sha256', 'test_sha256']].to_string(index=False))
else:
    print('[SKIP] Atmospheric control check: inventory is empty.')


## Step 2 — NetCDF Structure Comparison

For every file flagged DIFFERENT, compare variable lists, dimension sizes,
and data types.  Classify each variable as *physical*, *metadata*, or *unknown*.

> A checksum difference from metadata alone does **not** mean the physical
> state differs.  This step identifies which variables to actually analyse.


In [ ]:
%%time

classif_df = (structure_comparison_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    verbose=True,
) if IC_DATA_READY else pd.DataFrame())
if not IC_DATA_READY:
    print('[SKIP] Step 2: inventory is empty.')
if not classif_df.empty:
    display(classif_df.groupby(['component', 'classification', 'location'])
            .size().unstack(fill_value=0))


## Step 3 — Variable-Level IC Statistics

For every *physical* common variable in each DIFFERENT file, compute
area-weighted RMSE, MAD, mean difference, pattern correlation, and integral difference.
Also writes ΔX NetCDF files for Step 4.


In [ ]:
%%time

stats_df = (statistics_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    verbose=True,
) if IC_DATA_READY else pd.DataFrame())
if not IC_DATA_READY:
    print('[SKIP] Step 3: inventory is empty.')
if not stats_df.empty:
    display(
        stats_df.groupby('component')[['rmse', 'mad', 'pattern_corr']]
        .agg(['mean', 'max']).round(4)
    )


### Top variables by RMSE

In [ ]:
if not stats_df.empty:
    top = stats_df.nlargest(20, 'rmse')[
        ['component', 'variable', 'rmse', 'mad', 'pattern_corr', 'frac_differing']
    ].reset_index(drop=True)
    display(top)


## Step 4 — Spatial and Vertical Diagnostics

Generates global difference maps, zonal-mean profiles, vertical RMSE
profiles, and per-component RMSE bar charts.


In [ ]:
%%time

if IC_DATA_READY and not stats_df.empty:
    plotting_workflow.run(
        config_path=CONFIG_PATH,
        pilot_only=PILOT_ONLY,
        single_date=PILOT_DATE if PILOT_ONLY else None,
        top_n=15,
        figure_outdir=FIGURE_OUTDIR,
        verbose=True,
    )
else:
    print('[SKIP] Step 4: no variable statistics are available.')


### Inline map viewer

In [ ]:
from IPython.display import Image, display as ipy_display

pilot_maps = sorted(MAPS_DIR.glob(f"{ic_cfg.active_dates[0]}_*_diff_map.png")) if IC_DATA_READY else []
for p in pilot_maps[:6]:
    print(p.name)
    ipy_display(Image(str(p), width=800))


## Step 5 — Physical Consistency Checks

Cross-component interface checks run independently on each experiment:
* Atmosphere surface temperature vs land surface temperature
* Ocean SST vs sea-ice concentration (below-freezing cells)
* Simulation timestamp agreement across all components


In [ ]:
%%time

consistency_df = (physical_consistency_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    verbose=True,
) if IC_DATA_READY else pd.DataFrame())
if not IC_DATA_READY:
    print('[SKIP] Step 5: inventory is empty.')
if not consistency_df.empty:
    display(consistency_df[['check_name', 'experiment', 'frac_inconsistent', 'notes']])


## Step 6 — Campaign Summary

> ℹ️ This step requires data from all start dates.  Set `PILOT_ONLY = False`
> in the User Configuration cell and re-run Steps 1–5 before running this cell.


In [ ]:
%%time

campaign_df = (campaign_summary_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    verbose=True,
) if IC_DATA_READY and not PILOT_ONLY else pd.DataFrame())
if PILOT_ONLY:
    print('[SKIP] Step 6 requires PILOT_ONLY = False and completed full-campaign diagnostics.')
elif not IC_DATA_READY:
    print('[SKIP] Step 6: inventory is empty.')
if not campaign_df.empty:
    display(campaign_df.nlargest(10, 'mean_rmse'))


### Campaign heatmap

In [ ]:
from IPython.display import Image, display as ipy_display

heatmap_path = CS_DIR / 'campaign_summary.png'
if heatmap_path.exists():
    ipy_display(Image(str(heatmap_path), width=900))
else:
    print(f'Not found: {heatmap_path} -- run Step 6 with full campaign first.')


## Shutdown

In [ ]:
close_cluster(cluster, client)
